# SDA-CIA Data Ingestion

This notebook ingests monthly vehicle registration data from **SDA-CIA** (Svaz dovozců automobilů – Centrální registr vozidel) into the bronze layer.

## Source
- **Location:** `/Volumes/agentbricks/sector_data_raw/sda_cia_data`
- **Format:** xlsx files named `YYYY-M.monthly.CZ.xlsx` (one per month, from 2022-01 onwards)

## Tables Created

| Table | Description |
| --- | --- |
| `agentbricks.sector_data_bronze.sda_cia_oa_fuels_monthly` | New passenger car registrations by **brand** and **fuel type** per month (sheets: *OA Paliva za měsíc*, *OA Paliva za měsíc Jiné značk*) |
| `agentbricks.sector_data_bronze.sda_cia_oa_categories_monthly` | New passenger car registrations by **vehicle category/segment** per month (sheet: *OA Podíl tříd*) |

## Processing Logic
1. Parse all xlsx files from the source volume
2. Extract structured data from relevant sheets (unpivot fuel columns into rows)
3. Filter out zero-registration rows
4. Write to Delta tables with full overwrite (data is small, ~55k rows for fuels, ~500 rows for categories)
5. Add table and column comments for documentation

## Checks
- **Completeness check:** verifies all expected months (from earliest file to current date) are present in the output table
- **Summary statistics:** total rows, date range, distinct brands/fuel types, top brands, fuel type breakdown

In [0]:
%pip install openpyxl --quiet

In [0]:
# Configuration
SOURCE_PATH = "/Volumes/agentbricks/sector_data_raw/sda_cia_data"
TARGET_TABLE = "agentbricks.sector_data_bronze.sda_cia_oa_fuels_monthly"

# Sheets to ingest (main brands + other brands)
SHEET_NAMES = ["OA Paliva za měsíc", "OA Paliva za měsíc   Jiné značk"]

# Fuel type columns: (column_index, fuel_type_name)
# Based on row 2 header structure: values at col 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24
FUEL_COLUMNS = [
    (2, "Benzín"),
    (4, "Nafta"),
    (6, "CNG"),
    (8, "Nafta + CNG"),
    (10, "Benzín + CNG"),
    (12, "Benzín + LPG"),
    (14, "Vodík"),
    (16, "Elektro"),
    (18, "Benzín + El."),
    (20, "Nafta + El."),
    (22, "Jiné"),
    (24, "Nezařazeno"),
]

In [0]:
import os
import re
import openpyxl
from datetime import date
from dateutil.relativedelta import relativedelta
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType

def parse_filename(filename: str) -> tuple:
    """Extract year and month from filename like '2022-1.monthly.CZ.xlsx'"""
    match = re.match(r"(\d{4})-(\d{1,2})\.monthly\.CZ\.xlsx", filename)
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None


def parse_sheet(filepath: str, sheet_name: str, year: int, month: int) -> list:
    """
    Parse a fuel-by-brand sheet into structured rows.
    Returns list of tuples: (report_date, brand, fuel_type, registrations)
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    
    if sheet_name not in wb.sheetnames:
        wb.close()
        return []
    
    ws = wb[sheet_name]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()
    
    report_date = date(year, month, 1)
    results = []
    
    # Data rows start at index 4 (0-based), skip header/empty rows
    for row in rows[4:]:
        # First column is brand name
        brand = row[0]
        
        # Stop at summary rows (Celkem, Podíl, empty, or source notes)
        if brand is None or brand == '' or brand in ('Celkem', 'Podíl'):
            continue
        if 'Zdroj dat' in str(brand) or 'Pro SDA' in str(brand):
            continue
        
        brand = str(brand).strip()
        
        # Extract registration count for each fuel type
        for col_idx, fuel_type in FUEL_COLUMNS:
            try:
                val = row[col_idx]
                registrations = int(val) if val is not None else 0
            except (ValueError, TypeError, IndexError):
                registrations = 0
            
            results.append((report_date, brand, fuel_type, registrations))
    
    return results

print("Parsing functions defined.")

In [0]:
import time

# Get all source files
all_files = sorted([
    f for f in os.listdir(SOURCE_PATH)
    if f.endswith('.xlsx') and 'monthly.CZ' in f
])
print(f"Total files to process: {len(all_files)}")

# Parse all files
all_records = []
start_time = time.time()

for filename in all_files:
    year, month = parse_filename(filename)
    if year is None:
        continue
    
    filepath = os.path.join(SOURCE_PATH, filename)
    
    for sheet_name in SHEET_NAMES:
        records = parse_sheet(filepath, sheet_name, year, month)
        all_records.extend(records)
    
    elapsed = time.time() - start_time
    print(f"  {filename}: {len(all_records):,} total records so far ({elapsed:.1f}s)")

elapsed = time.time() - start_time
print(f"\nParsing complete: {len(all_records):,} records from {len(all_files)} files in {elapsed:.1f}s")

In [0]:
# Define schema and create Spark DataFrame
schema = StructType([
    StructField("report_date", DateType(), False),
    StructField("brand", StringType(), False),
    StructField("fuel_type", StringType(), False),
    StructField("registrations", IntegerType(), False),
])

df = spark.createDataFrame(all_records, schema=schema)

# Keep only rows with actual registrations
df = df.filter(df.registrations > 0)

print(f"DataFrame created: {df.count():,} rows (filtered to registrations > 0)")
print(f"\nSchema:")
df.printSchema()
print(f"\nSample data:")
display(df.orderBy("report_date", "brand", "fuel_type").limit(20))

In [0]:
# Write to Delta table (full overwrite - data is small and fast to recompute)
df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE)

print(f"\u2705 Table written: {TARGET_TABLE}")
print(f"   Rows: {spark.table(TARGET_TABLE).count():,}")

# Add table comment
spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS 
    'Monthly new passenger car (OA) registrations in Czech Republic by brand and fuel type. Source: SDA-CIA (Svaz dovozců automobilů - Centrální registr vozidel). Each row represents the number of new registrations for a specific brand and fuel type in a given month.'
""")

# Add column comments
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN report_date COMMENT 'First day of the reporting month (YYYY-MM-01)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN brand COMMENT 'Car manufacturer/brand name'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN fuel_type COMMENT 'Fuel/powertrain type: Benzín, Nafta, CNG, Elektro, Vodík, Benzín + El. (PHEV), Nafta + El., Benzín + LPG, Benzín + CNG, Nafta + CNG, Jiné, Nezařazeno'")
spark.sql(f"ALTER TABLE {TARGET_TABLE} ALTER COLUMN registrations COMMENT 'Number of new cars registered in the given month for this brand and fuel type'")

print("   Table and column comments added.")

In [0]:
# Completeness check: verify all months from earliest to latest are present
from pyspark.sql import functions as F

result_df = spark.table(TARGET_TABLE)

# Get distinct months in the data
months_in_data = set(
    row.report_date for row in 
    result_df.select("report_date").distinct().collect()
)

# Expected months from source files
expected_months = set()
for f in all_files:
    y, m = parse_filename(f)
    if y and m:
        expected_months.add(date(y, m, 1))

missing = expected_months - months_in_data
extra = months_in_data - expected_months

print("=" * 60)
print("COMPLETENESS CHECK")
print("=" * 60)
print(f"Expected months (from files): {len(expected_months)}")
print(f"Months in table: {len(months_in_data)}")

if missing:
    print(f"\n⚠️  MISSING months: {sorted(missing)}")
else:
    print("\n✓ All source months are present in the table.")

# Check current month gap
today = date.today()
current_month = date(today.year, today.month, 1)
latest_in_data = max(months_in_data)
print(f"\nLatest data: {latest_in_data}")
print(f"Current month: {current_month}")
if latest_in_data < current_month:
    gap_months = (current_month.year - latest_in_data.year) * 12 + (current_month.month - latest_in_data.month)
    print(f"⚠️  Data is {gap_months} month(s) behind current date (source files may not be available yet).")
else:
    print("✅ Data is up to date.")

In [0]:
# Summary statistics
from pyspark.sql import functions as F

result_df = spark.table(TARGET_TABLE)

print("=== Summary ===")
print(f"Total rows: {result_df.count():,}")
print(f"Date range: {result_df.agg(F.min('report_date')).collect()[0][0]} to {result_df.agg(F.max('report_date')).collect()[0][0]}")
print(f"Distinct brands: {result_df.select('brand').distinct().count()}")
print(f"Distinct fuel types: {result_df.select('fuel_type').distinct().count()}")

print("\n=== Registrations by fuel type (all time) ===")
display(
    result_df.groupBy("fuel_type")
    .agg(F.sum("registrations").alias("total_registrations"))
    .orderBy(F.desc("total_registrations"))
)

print("\n=== Top 10 brands by total registrations ===")
display(
    result_df.groupBy("brand")
    .agg(F.sum("registrations").alias("total_registrations"))
    .orderBy(F.desc("total_registrations"))
    .limit(10)
)

In [0]:
# Monthly trend: electric vs benzin vs nafta
from pyspark.sql import functions as F

trend_df = (
    spark.table(TARGET_TABLE)
    .filter(F.col("fuel_type").isin("Benzín", "Nafta", "Elektro", "Benzín + El."))
    .groupBy("report_date", "fuel_type")
    .agg(F.sum("registrations").alias("total"))
    .orderBy("report_date", "fuel_type")
)

display(trend_df)

In [0]:
# --- OA Podíl tříd: Car registrations by vehicle category ---
TARGET_TABLE_CATEGORIES = "agentbricks.sector_data_bronze.sda_cia_oa_categories_monthly"
CATEGORY_SHEET = "OA Podíl tříd"

# Known valid categories (row 4-13 in each file)
VALID_CATEGORIES = {
    "Nezařazeno", "Mini", "Malé", "Nižší střední", "Střední",
    "Vyšší střední", "Luxusní", "MPV", "Sportovní", "SUV a Terénní"
}

In [0]:
import time

def parse_category_sheet(filepath: str, year: int, month: int) -> list:
    """
    Parse 'OA Podíl tříd' sheet into structured rows.
    Returns list of tuples: (report_date, category, registrations)
    """
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    
    if CATEGORY_SHEET not in wb.sheetnames:
        wb.close()
        return []
    
    ws = wb[CATEGORY_SHEET]
    rows = list(ws.iter_rows(values_only=True))
    wb.close()
    
    report_date = date(year, month, 1)
    results = []
    
    # Data rows start at index 4
    for row in rows[4:]:
        category = row[0]
        if category is None or str(category).strip() == '':
            continue
        category = str(category).strip()
        if category not in VALID_CATEGORIES:
            continue
        
        try:
            registrations = int(row[2]) if row[2] is not None else 0
        except (ValueError, TypeError):
            registrations = 0
        
        if registrations > 0:
            results.append((report_date, category, registrations))
    
    return results

# Parse all files
category_records = []
start_time = time.time()

for filename in all_files:
    year, month = parse_filename(filename)
    if year is None:
        continue
    filepath = os.path.join(SOURCE_PATH, filename)
    records = parse_category_sheet(filepath, year, month)
    category_records.extend(records)

elapsed = time.time() - start_time
print(f"Parsing complete: {len(category_records):,} records from {len(all_files)} files in {elapsed:.1f}s")

In [0]:
# Create DataFrame and write to Delta
category_schema = StructType([
    StructField("report_date", DateType(), False),
    StructField("category", StringType(), False),
    StructField("registrations", IntegerType(), False),
])

df_cat = spark.createDataFrame(category_records, schema=category_schema)

df_cat.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TARGET_TABLE_CATEGORIES)

print(f"\u2705 Table written: {TARGET_TABLE_CATEGORIES}")
print(f"   Rows: {spark.table(TARGET_TABLE_CATEGORIES).count():,}")

# Add table and column comments
spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE_CATEGORIES} IS
    'Monthly new passenger car (OA) registrations in Czech Republic by vehicle category/segment. Source: SDA-CIA (Svaz dovozců automobilů - Centrální registr vozidel). Each row represents the total number of new registrations for a vehicle category in a given month.'
""")
spark.sql(f"ALTER TABLE {TARGET_TABLE_CATEGORIES} ALTER COLUMN report_date COMMENT 'First day of the reporting month (YYYY-MM-01)'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_CATEGORIES} ALTER COLUMN category COMMENT 'Vehicle segment/category: Mini, Malé, Nižší střední, Střední, Vyšší střední, Luxusní, MPV, Sportovní, SUV a Terénní, Nezařazeno'")
spark.sql(f"ALTER TABLE {TARGET_TABLE_CATEGORIES} ALTER COLUMN registrations COMMENT 'Number of new cars registered in the given month for this vehicle category'")

print("   Table and column comments added.")
print(f"\nSample:")
display(spark.table(TARGET_TABLE_CATEGORIES).orderBy("report_date", "category").limit(10))